In [1]:
# =============================================================================
# Phase 4 — Gold Layer ETL : Silver Featured → Gold
# Craigslist Used Vehicle Dataset
# =============================================================================

import logging
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
log = logging.getLogger("VehicleETL_Gold")



In [2]:
# =============================================================================
# CONFIG
# =============================================================================

# # Safeguard — ensure every table has a destination path
# assert len(GOLD_PATHS) == len({
#     "kpi_summary", "manufacturer_summary", "state_summary",
#     "monthly_summary", "fuel_summary", "segment_summary",
#     "condition_summary", "title_summary", "transmission_summary",
#     "state_manufacturer", "manufacturer_condition"
# }), "Mismatch between GOLD_PATHS and expected Gold tables."

SILVER_FEATURED_PATH = "hdfs:///user/vehicle_market/silver/vehicles_featured"

GOLD_BASE = "hdfs:///user/vehicle_market/gold"

GOLD_PATHS = {
    "kpi_summary"           : f"{GOLD_BASE}/kpi_summary",
    "manufacturer_summary"  : f"{GOLD_BASE}/manufacturer_summary",
    "state_summary"         : f"{GOLD_BASE}/state_summary",
    "monthly_summary"       : f"{GOLD_BASE}/monthly_summary",
    "fuel_summary"          : f"{GOLD_BASE}/fuel_summary",
    "segment_summary"       : f"{GOLD_BASE}/segment_summary",
    "condition_summary"     : f"{GOLD_BASE}/condition_summary",
    "title_summary"         : f"{GOLD_BASE}/title_summary",
    "transmission_summary"  : f"{GOLD_BASE}/transmission_summary",
    "state_manufacturer"    : f"{GOLD_BASE}/state_manufacturer",
    "manufacturer_condition": f"{GOLD_BASE}/manufacturer_condition",
}



In [3]:
# =============================================================================
# 1. SPARK SESSION
# =============================================================================

spark = SparkSession.builder \
    .appName("VehicleMarket_Gold") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
# spark.sparkContext.setCheckpointDir(
#     "hdfs:///user/vehicle_market/logs/checkpoints"
# )

pipeline_start = time.time()
log.info("="*65)
log.info("  Phase 4 — Gold Layer ETL : Silver Featured → Gold")
log.info("="*65)



26/07/20 00:38:32 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
2026-07-20 00:38:32 | INFO | =================================================================
2026-07-20 00:38:32 | INFO |   Phase 4 — Gold Layer ETL : Silver Featured → Gold
2026-07-20 00:38:32 | INFO | =================================================================


In [4]:
# =============================================================================
# 2. LOAD & CACHE SILVER FEATURED
# Caching here avoids repeated HDFS reads across all 11 aggregations
# =============================================================================

log.info("Loading and caching Silver Featured layer...")
df = spark.read.parquet(SILVER_FEATURED_PATH)
df.cache()

# Add right after the df.count() line in section 2
log.info("Silver Featured Schema:")
df.printSchema()

# print("Columns in Silver Featured:")
# for col in sorted(df.columns):
#     print(col)



record_count_in = df.count()          # triggers cache materialization
log.info(f"  Records loaded and cached : {record_count_in}")
log.info(f"  Columns                   : {len(df.columns)}")



2026-07-20 00:38:32 | INFO | Loading and caching Silver Featured layer...
26/07/20 00:38:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
2026-07-20 00:38:34 | INFO | Silver Featured Schema:


root
 |-- manufacturer: string (nullable = true)
 |-- id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- region: string (nullable = true)
 |-- region_url: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- model: string (nullable = true)
 |-- condition: string (nullable = true)
 |-- cylinders: string (nullable = true)
 |-- fuel: string (nullable = true)
 |-- odometer: integer (nullable = true)
 |-- title_status: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- VIN: string (nullable = true)
 |-- drive: string (nullable = true)
 |-- size: string (nullable = true)
 |-- type: string (nullable = true)
 |-- paint_color: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- description: string (nullable = true)
 |-- state: string (nullable = true)
 |-- lat: float (nullable = true)
 |-- long: float (nullable = true)
 |-- posting_date: timestamp (nullable = true)
 |-- vehicle_age: i

2026-07-20 00:38:37 | INFO |   Records loaded and cached : 242979               
2026-07-20 00:38:37 | INFO |   Columns                   : 45


In [5]:
# =============================================================================
# 3. DATA QUALITY VALIDATION
# =============================================================================

log.info("Running pre-aggregation data quality validation...")

REQUIRED_COLS = [
    "manufacturer", "state", "price", "odometer", "fuel",
    "transmission", "condition", "title_status", "vehicle_age",
    "age_group", "price_category", "mileage_category", "vehicle_segment",
    "posting_year", "posting_month", "posting_season", "is_luxury",
    "clean_title", "is_salvage", "is_automatic", "price_band"
]

missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
if missing_cols:
    log.error(f"  Missing required columns: {missing_cols}")
    raise ValueError(f"Silver Featured is missing columns: {missing_cols}")

log.info("  All required columns present.")

for col in ["price", "year", "manufacturer", "state", "odometer"]:
    null_count = df.filter(F.col(col).isNull()).count()
    null_pct   = null_count / record_count_in * 100
    status     = "✓" if null_pct == 0 else "⚠"
    log.info(f"  {status} {col:<20} nulls: {null_count} ({null_pct:.2f}%)")

log.info("  Data quality validation complete.")



2026-07-20 00:38:37 | INFO | Running pre-aggregation data quality validation...
2026-07-20 00:38:37 | INFO |   All required columns present.
2026-07-20 00:38:37 | INFO |   ✓ price                nulls: 0 (0.00%)
2026-07-20 00:38:37 | INFO |   ✓ year                 nulls: 0 (0.00%)
2026-07-20 00:38:37 | INFO |   ✓ manufacturer         nulls: 0 (0.00%)
2026-07-20 00:38:37 | INFO |   ✓ state                nulls: 0 (0.00%)
2026-07-20 00:38:37 | INFO |   ✓ odometer             nulls: 0 (0.00%)
2026-07-20 00:38:37 | INFO |   Data quality validation complete.


In [6]:
# =============================================================================
# HELPER 1 — Standard price + inventory aggregations
# Used across all Gold tables to avoid repeated code
# =============================================================================

def price_aggs():
    return [
        F.count("*").alias("total_listings"),
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.round(F.percentile_approx("price", 0.5), 2).alias("median_price"),
        F.min("price").alias("min_price"),
        F.max("price").alias("max_price"),
        F.round(F.stddev("price"), 2).alias("stddev_price"),
        F.round(F.avg("odometer"), 0).alias("avg_mileage"),
        F.round(F.avg("vehicle_age"), 1).alias("avg_vehicle_age"),
    ]



In [7]:
# =============================================================================
# HELPER 2 — Write Gold table with count-before-write + empty check
# =============================================================================

def write_gold(table_df, table_name):
    path      = GOLD_PATHS[table_name]
    row_count = table_df.count()           # count before write

    if row_count == 0:
        raise ValueError(
            f"[{table_name}] produced 0 rows — aborting write. "
            f"Check source data or groupBy keys."
        )

    table_df.write.mode("overwrite").parquet(path)
    log.info(f"  ✓ {table_name:<30} rows: {row_count:<8} → {path}")
    return row_count



In [8]:
# =============================================================================
# 4. GOLD TABLE 1 — EXECUTIVE KPI SUMMARY
# Single-row table for all Dashboard 1 KPI cards:
# Total Listings, Avg Price, Avg Mileage, Avg Age, Manufacturers count
# =============================================================================

log.info("Building kpi_summary...")

kpi_summary = df.agg(
    F.count("*").alias("total_listings"),
    F.round(F.avg("price"), 2).alias("avg_price"),
    F.round(F.percentile_approx("price", 0.5), 2).alias("median_price"),
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.round(F.avg("odometer"), 0).alias("avg_mileage"),
    F.round(F.avg("vehicle_age"), 1).alias("avg_vehicle_age"),
    F.countDistinct("manufacturer").alias("total_manufacturers"),
    F.countDistinct("state").alias("total_states"),
    F.countDistinct("model").alias("total_models"),
    F.round(
        F.sum(F.col("clean_title").cast("int")) /
        F.count("*") * 100, 2
    ).alias("clean_title_pct"),
    F.round(
        F.sum(F.col("is_luxury").cast("int")) /
        F.count("*") * 100, 2
    ).alias("luxury_pct"),
    F.round(
        F.sum(F.col("is_salvage").cast("int")) /
        F.count("*") * 100, 2
    ).alias("salvage_pct"),
    F.round(
        F.sum(F.col("is_alternative_fuel").cast("int")) /
        F.count("*") * 100, 2
    ).alias("alt_fuel_pct"),
    F.round(
        F.sum(F.col("is_automatic").cast("int")) /
        F.count("*") * 100, 2
    ).alias("automatic_pct")
)



2026-07-20 00:38:37 | INFO | Building kpi_summary...


In [9]:
# =============================================================================
# 5. GOLD TABLE 2 — MANUFACTURER SUMMARY
# Dashboards: Dashboard 1 (Top Manufacturers bar),
#             Dashboard 2 (Avg Price by Manufacturer, heatmap),
#             Dashboard 4 (Manufacturer Popularity by State)
# =============================================================================

log.info("Building manufacturer_summary...")

manufacturer_summary = df.groupBy("manufacturer", "is_luxury") \
    .agg(
        *price_aggs(),
        F.round(F.avg("price_per_mile"), 4).alias("avg_price_per_mile"),
        F.countDistinct("model").alias("distinct_models"),
        F.round(
            F.sum(F.col("clean_title").cast("int")) /
            F.count("*") * 100, 2
        ).alias("clean_title_pct"),
        F.round(
            F.sum(F.col("is_salvage").cast("int")) /
            F.count("*") * 100, 2
        ).alias("salvage_pct"),
        F.round(
            F.sum(F.col("is_alternative_fuel").cast("int")) /
            F.count("*") * 100, 2
        ).alias("alt_fuel_pct")
    ).orderBy(F.desc("total_listings"))



2026-07-20 00:38:37 | INFO | Building manufacturer_summary...


In [10]:
# =============================================================================
# 6. GOLD TABLE 3 — STATE SUMMARY
# Dashboards: Dashboard 1 (USA Map), Dashboard 4 (Geographic Analytics,
#             State Rankings, Mileage by State)
# =============================================================================

log.info("Building state_summary...")

state_summary = df.groupBy("state") \
    .agg(
        *price_aggs(),
        F.round(F.avg("lat"), 6).alias("avg_lat"),
        F.round(F.avg("long"), 6).alias("avg_long"),
        F.round(
            F.sum(F.col("is_luxury").cast("int")) /
            F.count("*") * 100, 2
        ).alias("luxury_pct"),
        F.round(
            F.sum(F.col("is_alternative_fuel").cast("int")) /
            F.count("*") * 100, 2
        ).alias("alt_fuel_pct"),
        F.round(
            F.sum(F.col("clean_title").cast("int")) /
            F.count("*") * 100, 2
        ).alias("clean_title_pct")
    ).orderBy(F.desc("total_listings"))



2026-07-20 00:38:37 | INFO | Building state_summary...


In [11]:
# =============================================================================
# 7. GOLD TABLE 4 — MONTHLY SUMMARY
# Dashboards: Dashboard 1 (Listing Trend line chart),
#             Dashboard 5 (Live Trend baseline)
# =============================================================================

log.info("Building monthly_summary...")

monthly_summary = df.groupBy(
    "posting_year", "posting_month",
    "posting_month_name", "posting_quarter", "posting_season"
).agg(
    *price_aggs(),
    F.round(
        F.sum(F.col("is_luxury").cast("int")) /
        F.count("*") * 100, 2
    ).alias("luxury_pct"),
    F.round(
        F.sum(F.col("is_alternative_fuel").cast("int")) /
        F.count("*") * 100, 2
    ).alias("alt_fuel_pct")
).orderBy("posting_year", "posting_month")



2026-07-20 00:38:37 | INFO | Building monthly_summary...


In [12]:
# =============================================================================
# 8. GOLD TABLE 5 — FUEL SUMMARY
# Dashboards: Dashboard 1 (Fuel Distribution donut),
#             Dashboard 2 (Price by Fuel Type)
# =============================================================================

log.info("Building fuel_summary...")

fuel_summary = df.groupBy("fuel", "is_alternative_fuel") \
    .agg(
        *price_aggs(),
        F.round(
            F.count("*") / F.lit(record_count_in) * 100, 2
        ).alias("listing_pct")
    ).orderBy(F.desc("total_listings"))



2026-07-20 00:38:37 | INFO | Building fuel_summary...


In [13]:
# =============================================================================
# 9. GOLD TABLE 6 — VEHICLE SEGMENT SUMMARY
# Dashboards: Dashboard 3 (Vehicle Type treemap, Inventory analysis)
# Note: Replaces type, drive, age_group, mileage as separate Gold tables.
#       Tableau can slice those directly from Silver Featured or this table.
# =============================================================================

log.info("Building segment_summary...")

segment_summary = df.groupBy("vehicle_segment") \
    .agg(
        *price_aggs(),
        F.round(
            F.count("*") / F.lit(record_count_in) * 100, 2
        ).alias("listing_pct"),
        F.round(
            F.sum(F.col("clean_title").cast("int")) /
            F.count("*") * 100, 2
        ).alias("clean_title_pct"),
        F.round(
            F.sum(F.col("is_luxury").cast("int")) /
            F.count("*") * 100, 2
        ).alias("luxury_pct")
    ).orderBy(F.desc("total_listings"))



2026-07-20 00:38:37 | INFO | Building segment_summary...


In [14]:
# =============================================================================
# 10. GOLD TABLE 7 — CONDITION SUMMARY
# Dashboards: Dashboard 1 (Vehicle Condition stacked bar),
#             Dashboard 2 (Price by Condition, heatmap)
# =============================================================================

log.info("Building condition_summary...")

condition_summary = df.groupBy("condition") \
    .agg(
        *price_aggs(),
        F.round(
            F.count("*") / F.lit(record_count_in) * 100, 2
        ).alias("listing_pct")
    ).orderBy(F.desc("total_listings"))



2026-07-20 00:38:37 | INFO | Building condition_summary...


In [15]:
# =============================================================================
# 11. GOLD TABLE 8 — TITLE STATUS SUMMARY
# Dashboards: Dashboard 3 (Title Status, Clean Title % KPI, Salvage % KPI)
# =============================================================================

log.info("Building title_summary...")

title_summary = df.groupBy("title_status", "clean_title", "is_salvage") \
    .agg(
        *price_aggs(),
        F.round(
            F.count("*") / F.lit(record_count_in) * 100, 2
        ).alias("listing_pct")
    ).orderBy(F.desc("total_listings"))



2026-07-20 00:38:37 | INFO | Building title_summary...


In [16]:
# =============================================================================
# 12. GOLD TABLE 9 — TRANSMISSION SUMMARY
# Dashboards: Dashboard 2 (Price by Transmission),
#             Dashboard 3 (Automatic % KPI)
# =============================================================================

log.info("Building transmission_summary...")

transmission_summary = df.groupBy("transmission") \
    .agg(
        *price_aggs(),
        F.round(
            F.count("*") / F.lit(record_count_in) * 100, 2
        ).alias("listing_pct")
    ).orderBy(F.desc("total_listings"))



2026-07-20 00:38:37 | INFO | Building transmission_summary...


In [17]:
# =============================================================================
# 13. GOLD TABLE 10 — STATE × MANUFACTURER HEATMAP
# Dashboards: Dashboard 4 (Manufacturer Popularity by State heatmap)
# =============================================================================

log.info("Building state_manufacturer heatmap...")

state_manufacturer = df.groupBy("state", "manufacturer") \
    .agg(
        F.count("*").alias("total_listings"),
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.round(F.avg("odometer"), 0).alias("avg_mileage")
    ).orderBy(F.desc("total_listings"))



2026-07-20 00:38:38 | INFO | Building state_manufacturer heatmap...


In [18]:
# =============================================================================
# 14. GOLD TABLE 11 — MANUFACTURER × CONDITION HEATMAP
# Dashboards: Dashboard 2 (Manufacturer × Condition × Avg Price heatmap)
# =============================================================================

log.info("Building manufacturer_condition heatmap...")

manufacturer_condition = df.groupBy("manufacturer", "condition") \
    .agg(
        F.count("*").alias("total_listings"),
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.round(F.percentile_approx("price", 0.5), 2).alias("median_price"),
        F.round(F.avg("vehicle_age"), 1).alias("avg_vehicle_age")
    ).orderBy(F.desc("total_listings"))



2026-07-20 00:38:38 | INFO | Building manufacturer_condition heatmap...


In [19]:
# =============================================================================
# 15. WRITE ALL GOLD TABLES
# =============================================================================

log.info("Writing Gold tables to HDFS...")

gold_tables = {
    "kpi_summary"           : kpi_summary,
    "manufacturer_summary"  : manufacturer_summary,
    "state_summary"         : state_summary,
    "monthly_summary"       : monthly_summary,
    "fuel_summary"          : fuel_summary,
    "segment_summary"       : segment_summary,
    "condition_summary"     : condition_summary,
    "title_summary"         : title_summary,
    "transmission_summary"  : transmission_summary,
    "state_manufacturer"    : state_manufacturer,
    "manufacturer_condition": manufacturer_condition,
}

# Add this right before the write loop in section 15
missing_paths = [name for name in gold_tables if name not in GOLD_PATHS]
if missing_paths:
    raise ValueError(f"Missing GOLD_PATHS entries for: {missing_paths}")
log.info(f"  Path validation passed — {len(gold_tables)} tables, {len(GOLD_PATHS)} paths.")

write_results = {}
for table_name, table_df in gold_tables.items():
    try:
        row_count = write_gold(table_df, table_name)
        write_results[table_name] = {"status": "✓", "rows": row_count}
    except Exception as e:
        write_results[table_name] = {"status": "✗", "rows": 0}
        log.error(f"  ✗ {table_name} FAILED: {e}")



2026-07-20 00:38:38 | INFO | Writing Gold tables to HDFS...
2026-07-20 00:38:38 | INFO |   Path validation passed — 11 tables, 11 paths.
Java HotSpot(TM) 64-Bit Server VM warning: CodeCache is full. Compiler has been disabled.
Java HotSpot(TM) 64-Bit Server VM warning: Try increasing the code cache size using -XX:ReservedCodeCacheSize=


CodeCache: size=131072Kb used=34853Kb max_used=34853Kb free=96218Kb
 bounds [0x000000010468c000, 0x00000001068cc000, 0x000000010c68c000]
 total_blobs=13453 nmethods=12385 adapters=980
 compilation: disabled (not enough contiguous free space left)


2026-07-20 00:38:39 | INFO |   ✓ kpi_summary                    rows: 1        → hdfs:///user/vehicle_market/gold/kpi_summary
2026-07-20 00:38:42 | INFO |   ✓ manufacturer_summary           rows: 43       → hdfs:///user/vehicle_market/gold/manufacturer_summary
2026-07-20 00:38:43 | INFO |   ✓ state_summary                  rows: 51       → hdfs:///user/vehicle_market/gold/state_summary
2026-07-20 00:38:44 | INFO |   ✓ monthly_summary                rows: 2        → hdfs:///user/vehicle_market/gold/monthly_summary
2026-07-20 00:38:44 | INFO |   ✓ fuel_summary                   rows: 6        → hdfs:///user/vehicle_market/gold/fuel_summary
2026-07-20 00:38:45 | INFO |   ✓ segment_summary                rows: 7        → hdfs:///user/vehicle_market/gold/segment_summary
2026-07-20 00:38:45 | INFO |   ✓ condition_summary              rows: 7        → hdfs:///user/vehicle_market/gold/condition_summary
2026-07-20 00:38:46 | INFO |   ✓ title_summary                  rows: 7        → hdfs:///use

In [20]:
# =============================================================================
# 16. UNPERSIST CACHE
# =============================================================================

df.unpersist()
log.info("Silver Featured cache released.")



2026-07-20 00:38:48 | INFO | Silver Featured cache released.


In [21]:
# =============================================================================
# 17. GOLD LAYER VALIDATION REPORT
# =============================================================================

pipeline_end   = time.time()
execution_time = round(pipeline_end - pipeline_start, 2)

tables_passed = sum(1 for v in write_results.values() if v["status"] == "✓")
tables_failed = sum(1 for v in write_results.values() if v["status"] == "✗")

log.info("\n" + "="*65)
log.info("         GOLD LAYER — VALIDATION REPORT")
log.info("="*65)
log.info(f"  Execution Time          : {execution_time}s")
log.info(f"  Records In (Featured)   : {record_count_in}")
log.info(f"  Gold Tables Written     : {tables_passed} / {len(gold_tables)}")
log.info(f"  Tables Failed           : {tables_failed}")
log.info("")
log.info(f"  {'Table':<30} {'Status':>6}  {'Rows':>8}")
log.info(f"  {'-'*48}")

for table_name, result in write_results.items():
    log.info(
        f"  {result['status']}  {table_name:<30} {result['rows']:>8} rows"
    )

log.info("")
log.info("  KPI Snapshot:")
kpi_summary.show(1, truncate=False, vertical=True)

log.info("")
if tables_failed == 0:
    log.info("  ✓ All Gold tables written successfully.")
    log.info("  ✓ Ready for Hive Data Warehouse.")
else:
    log.info(f"  ⚠ {tables_failed} table(s) failed. Review errors above.")

log.info("="*65)

2026-07-20 00:38:48 | INFO | 
2026-07-20 00:38:48 | INFO |          GOLD LAYER — VALIDATION REPORT
2026-07-20 00:38:48 | INFO | =================================================================
2026-07-20 00:38:48 | INFO |   Execution Time          : 16.25s
2026-07-20 00:38:48 | INFO |   Records In (Featured)   : 242979
2026-07-20 00:38:48 | INFO |   Gold Tables Written     : 11 / 11
2026-07-20 00:38:48 | INFO |   Tables Failed           : 0
2026-07-20 00:38:48 | INFO | 
2026-07-20 00:38:48 | INFO |   Table                          Status      Rows
2026-07-20 00:38:48 | INFO |   ------------------------------------------------
2026-07-20 00:38:48 | INFO |   ✓  kpi_summary                           1 rows
2026-07-20 00:38:48 | INFO |   ✓  manufacturer_summary                 43 rows
2026-07-20 00:38:48 | INFO |   ✓  state_summary                        51 rows
2026-07-20 00:38:48 | INFO |   ✓  monthly_summary                       2 rows
2026-07-20 00:38:48 | INFO |   ✓  fuel_summary   

-RECORD 0-----------------------
 total_listings      | 242979   
 avg_price           | 15365.38 
 median_price        | 10999    
 min_price           | 100      
 max_price           | 150000   
 avg_mileage         | 107376.0 
 avg_vehicle_age     | 15.4     
 total_manufacturers | 43       
 total_states        | 51       
 total_models        | 24299    
 clean_title_pct     | 93.96    
 luxury_pct          | 14.97    
 salvage_pct         | 3.99     
 alt_fuel_pct        | 4.78     
 automatic_pct       | 88.67    

